# Valeric Acid

In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
valeric,102.13,3.89077,3.409695202,236.0879235,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
valeric,H,valeric,e,2000.0,0.03
"""

model = PCSAFT(["valeric"], userlocations = [like_parameter, assoc_parameter])

PCSAFT{BasicIdeal, Float64} with 1 component:
 "valeric"
Contains parameters: Mw, segment, sigma, epsilon, epsilon_assoc, bondvol

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_valeric.csv")
fix_line_endings("rhol_valeric.csv")

Fixed: satp_valeric.csv
Fixed: rhol_valeric.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end 

my_liquid_density (generic function with 1 method)

In [5]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 500.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.001,
        :upper   => 0.50,
        :guess   => 0.4
    ),
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 500.0)
 Dict(:upper => 0.5, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.001)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_valeric.csv",
        "satp_valeric.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.6650771471093356


In [7]:
method = ECA(; options = Options(iterations = 10000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([2475.3781081010634, 0.048068268093113306], PCSAFT{BasicIdeal, Float64}("valeric"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_valeric.csv",   my_saturation_p)


=== AAD: satp_valeric.csv ===


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


Clapeyron Estimator  exp           calc          ARD%    
345.5500    1410.525600   1433.531261   1.6310  
345.7500    1423.857600   1450.412125   1.8650  
351.5500    1986.468000   2022.217510   1.7997  
363.1500    3636.969600   3779.314704   3.9138  
372.3500    6122.054400   5995.833815   2.0617  
378.9500    9591.040800   8205.916655   14.4419 
446.8500    101323.200000  106339.698066  4.9510  
AARD = 4.3806%


4.380576663532616

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_valeric.csv",   my_liquid_density)


=== AAD: rhol_valeric.csv ===
Clapeyron Estimator  exp           calc          ARD%    
273.1500    957.440000    932.112455    2.6453  
288.1500    943.740000    919.739592    2.5431  
303.1500    930.170000    907.544538    2.4324  
291.3000    941.100000    917.167480    2.5430  
292.2500    939.700000    916.393146    2.4802  
AARD = 2.5288%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


2.5288277029756103